# 브루트 포스 (Brute Force)

`-` 가능한 모든 경우의 수를 탐색하자

## Moo Hunt

- 문제 출처: [정올 8929번](https://jungol.co.kr/problem/8929)

`-` 가능한 모든 게임 칸마다 각 행동이 점수를 얻을 수 있는지 확인하는 브루트 포스 알고리즘은 $O\left(K 2^N\right)$의 시간 복잡도를 가진다

`-` $N$은 최대 $20$이고 $K$는 최대 $200000$이므로 시간 초과이다. 최적화가 필요하다 

`-` 비둘기 집 원리에 의해 $K$번의 행동 중 겹치는 것이 존재한다. $y$와 $z$는 서로 바뀌어도 무방하므로 중요한 건 $x$이다. 따라서 가능한 행동의 수는 $\frac{N(N-1)(N-2)}{2}$이다

`-` 이 경우 시간 복잡도는 $O\left(N^3 2^N\right)$이 된다

`-` 전보다는 나아졌지만 $N$이 최대 $20$이므로 제한 시간 안에 통과할 수 없다

`-` 점수를 획득하기 위해선 $x$가 $M$이어야 한다. 가능한 모든 행동에 대해 $x$가 동일한 것끼리 묶자. 보드가 정해진 뒤 각 $i$번 칸에 대해 $M$일 때만 $x=i$인 행동들의 점수를 계산하자

`-` 마찬가지로 동일한 $x$에 대해 동일한 $y$를 가진 행동끼리 묶자. 그럼 $y$번 칸이 $O$가 아닌 경우 $z$번 칸들은 확인할 필요도 없다

`-` 또한 남은 모든 행동이 점수를 획득해도 최대 점수 미만이면 탐색을 중지하자. 이를 위해 빈도수가 높은 행동부터 고려할 것이다

`-` 이러한 가지치기에 의해 각 보드별로 실제 확인할 행동의 수가 많이 줄어들게 된다

In [8]:
from itertools import product


def maximize_score(actions, x_counts, xy_counts, k):
    n = len(actions)
    boards = product(["M", "O"], repeat=n)
    max_score, count = 0, 0
    x_candidates = sorted(range(n), key=lambda x: -x_counts[x])
    for board in boards:
        score = 0
        remaining_attempts = k
        for x in x_candidates:
            if board[x] != "M":
                remaining_attempts -= x_counts[x]
                continue
            if score + remaining_attempts < max_score:
                break
            y_counts = xy_counts[x]
            for y in range(n):
                if board[y] != "O":
                    remaining_attempts -= y_counts[y]
                    continue
                for z, c in enumerate(actions[x][y]):
                    if c == 0:
                        continue
                    if board[z] == "O":
                        score += c
                    remaining_attempts -= 1
        if score > max_score:
            max_score = score
            count = 1
        elif score == max_score:
            count += 1
    return max_score, count


def solution():
    N, K = map(int, input().split())
    actions = [[[0] * i for i in range(N)] for _ in range(N)]
    x_counts = [0] * N
    xy_counts = [[0] * N for _ in range(N)] 
    for _ in range(K):
        x, y, z = map(lambda x: int(x) - 1, input().split())
        if y < z:
            y, z = z, y
        actions[x][y][z] += 1
        x_counts[x] += 1
        xy_counts[x][y] += 1
    max_score, count = maximize_score(actions, x_counts, xy_counts, K)
    print(max_score, count)


solution()

# input
# 3 1
# 1 2 3

 3 1
 1 2 3


1 1


## 못생긴 수

- 문제 출처: [정올 1318번](https://jungol.co.kr/problem/1318)

`-` 처음 봤을 때 뭔가 풀기 애매해 스킵했던 문제. 런타임 전의 전처리를 활용해서 풀 수 있다 (아예 깡으로 $1500$개를 하드코딩 해도 된다)

`-` $1500$번째 못생긴 수는 $2^{30}$을 넘지 않는다. 따라서 $1500$번째 이전의 못생긴 수 $x$에 대해 $x=2^i 3^j 5^k,\; (0 \le i \le 30,\, 0 \le j \le 19,\, 0 \le k \le 12)$로 정의된다

`-` $8060$개의 못생긴 수를 계산한 뒤 정렬하자. 쿼리가 들어오면 인덱싱만 해서 출력하면 된다 

In [13]:
def solution():
    max_n = 1500
    ugly_nums = []
    for i in range(31):
        for j in range(20):
            for k in range(13):
                x = 2**i * 3**j * 5**k
                ugly_nums.append(x)
    ugly_nums.sort()
    while True:
        n = int(input())
        if n == 0:
            break
        print(ugly_nums[n - 1])


solution()

# input
# 1
# 2
# 5
# 0

 1


1


 2


2


 5


5


 0


`-` 우선순위 큐를 이용해서 아름답게 해결할 수 있다. 초기 $1$부터 시작해 $2,3,5$를 곱해서 힙에 넣자. 그리고 최솟값을 뽑고 따로 배열에 추가한 뒤 $2,3,5$를 곱하고 힙에 넣는 걸 반복하면 된다